# Checkpoint probe (CPU only, no GPU quota)

`dreaddevelopment` has published several CoAtNet checkpoints, and v6 currently uses
`raptor_ft_coatnet384x.pt` (2026-08-21). Newer ones exist (v5 SWA 08-22, v4 + SWA 08-23,
v9 08-23). The training script stores `gold_auc` and per-target `aucs` inside every
checkpoint, so all of them can be ranked on the held-out 58-study gate by reading headers
alone -- no inference, no GPU, no training.

In [ ]:
import glob, os, torch, json
import numpy as np
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA",
       "Effusion","Synovitis","Baker's","Contusion","Fracture"]
pts = []
for _r, _d, _f in os.walk('/kaggle/input'):
    _d[:] = [x for x in _d if x not in ('train_series','test_series')]
    pts += [os.path.join(_r, x) for x in _f if x.endswith('.pt')]
pts = sorted(pts)
print(f'found {len(pts)} checkpoints\n')
rows=[]
for p in pts:
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
    except Exception as e:
        print(f'  {os.path.basename(p):34s} UNREADABLE {type(e).__name__}'); continue
    if not isinstance(ck, dict): 
        print(f'  {os.path.basename(p):34s} not a dict'); continue
    keys = [k for k in ck if k != 'model']
    row = {'file': os.path.basename(p),
           'dataset': os.path.relpath(os.path.dirname(p), '/kaggle/input'),
           'gold_auc': ck.get('gold_auc'), 'arch': ck.get('arch'), 'res': ck.get('res'),
           'src': ck.get('src'), 'tag': ck.get('tag'), 'n_train': ck.get('n_train')}
    rows.append(row)
    print(f"  {row['file']:34s} gold_auc={str(row['gold_auc'])[:8]:8s} res={row['res']} "
          f"arch={str(row['arch'])[:38]}")
    if isinstance(ck.get('aucs'), (list, np.ndarray)) and len(ck['aucs'])==12:
        row['aucs']=[float(x) for x in ck['aucs']]
    del ck
json.dump(rows, open('/kaggle/working/ckpt_probe.json','w'), indent=2, default=str)
print()
have=[r for r in rows if isinstance(r.get('gold_auc'),(int,float))]
have.sort(key=lambda r: -r['gold_auc'])
print('ranked by recorded gold-gate AUC (58 held-out studies):')
for r in have:
    print(f"  {r['gold_auc']:.4f}  {r['file']:34s} ({r['dataset']})")

In [ ]:
# Per-target detail for the top checkpoints, to see where any gain lands.
import json, numpy as np
rows=json.load(open('/kaggle/working/ckpt_probe.json'))
LAB = ["ACL","MCL","Medial Meniscus","Lateral Meniscus","Medial OA","Lateral OA","PF OA",
       "Effusion","Synovitis","Baker's","Contusion","Fracture"]
have=[r for r in rows if isinstance(r.get('gold_auc'),(int,float)) and r.get('aucs')]
have.sort(key=lambda r:-r['gold_auc'])
if have:
    print(f"{'target':18s}" + ''.join(f"{r['file'][:14]:>16s}" for r in have[:4]))
    for i,t in enumerate(LAB):
        print(f'{t:18s}' + ''.join(f"{r['aucs'][i]:16.4f}" for r in have[:4]))
    print(f"{'MACRO':18s}" + ''.join(f"{r['gold_auc']:16.4f}" for r in have[:4]))
else:
    print('no per-target aucs stored')